In [ ]:
# =============================================================================
# ERASMUS MOBILITY DATA: EUROSTAT REGIONAL SOCIOECONOMIC ENRICHMENT
# =============================================================================
# 
# This notebook enriches the dataset with regional socioeconomic indicators
# from Eurostat, matched by NUTS region and year (2014-2023).
#
# INDICATORS ADDED:
# - GDP per capita (NUTS3, NUTS2, Country levels)
# - Crime rates (NUTS2)
# - Population (NUTS3)
#
# INPUT:  erasmus_with_eter.csv (output from notebook 03)
# OUTPUT: erasmus_final_complete.csv - fully enriched dataset
#
# REQUIREMENTS:
# - eurostat Python package (pip install eurostat)
# - Internet connection for initial Eurostat downloads
#
# ESTIMATED RUNTIME: 20-30 minutes (first run with downloads)
# =============================================================================

In [ ]:
# =============================================================================
# CELL 1: Import Libraries
# =============================================================================

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully!")
print(f"  pandas version: {pd.__version__}")

In [ ]:
# =============================================================================
# CELL 2: Configuration - USER INPUT REQUIRED
# =============================================================================

# ==========================
# CONFIGURE THESE PATHS:
# ==========================

# Path to ETER-enriched data (output from notebook 03)
ETER_ENRICHED_PATH = '/path/to/erasmus_with_eter.csv'

# Output directory for final data and raw Eurostat files
OUTPUT_DIR = '/path/to/output/directory/'

# ==========================
# EUROSTAT CONFIGURATION:
# ==========================

# Download delay between datasets (seconds)
EUROSTAT_DELAY = 2

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(f"Input data:  {ETER_ENRICHED_PATH}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Eurostat delay: {EUROSTAT_DELAY}s between downloads")
print("=" * 80)

In [ ]:
# =============================================================================
# CELL 3: Install and Import Eurostat Package
# =============================================================================

print("\n" + "=" * 80)
print("SETTING UP EUROSTAT PACKAGE")
print("=" * 80)

try:
    import eurostat
    print("✓ eurostat package available")
except ImportError:
    print("Installing eurostat package...")
    import subprocess
    subprocess.run(['pip', 'install', 'eurostat'], check=True)
    import eurostat
    print("✓ eurostat package installed")

In [ ]:
# =============================================================================
# CELL 4: Download Eurostat Datasets
# =============================================================================

print("\n" + "=" * 80)
print("DOWNLOADING EUROSTAT DATASETS")
print("=" * 80)
print("This may take 15-20 minutes on first run...")
print("Raw files will be saved for future use\n")

# ============================================================================
# Dataset 1: GDP at NUTS3 level
# ============================================================================
print("1. GDP at NUTS3 level (nama_10r_3gdp)...")
try:
    gdp_nuts3 = eurostat.get_data_df('nama_10r_3gdp')
    print(f"   ✓ Downloaded {len(gdp_nuts3):,} rows")
    
    gdp_nuts3_path = f"{OUTPUT_DIR}/eurostat_gdp_nuts3_raw.csv"
    gdp_nuts3.to_csv(gdp_nuts3_path, index=False)
    print(f"   ✓ Saved: eurostat_gdp_nuts3_raw.csv")
    
except Exception as e:
    print(f"   ✗ Error: {e}")

time.sleep(EUROSTAT_DELAY)

# ============================================================================
# Dataset 2: GDP at NUTS2 level
# ============================================================================
print("\n2. GDP at NUTS2 level (nama_10r_2gdp)...")
try:
    gdp_nuts2 = eurostat.get_data_df('nama_10r_2gdp')
    print(f"   ✓ Downloaded {len(gdp_nuts2):,} rows")
    
    gdp_nuts2_path = f"{OUTPUT_DIR}/eurostat_gdp_nuts2_raw.csv"
    gdp_nuts2.to_csv(gdp_nuts2_path, index=False)
    print(f"   ✓ Saved: eurostat_gdp_nuts2_raw.csv")
    
except Exception as e:
    print(f"   ✗ Error: {e}")

time.sleep(EUROSTAT_DELAY)

# ============================================================================
# Dataset 3: GDP at Country level
# ============================================================================
print("\n3. GDP at country level (nama_10_gdp)...")
try:
    gdp_country = eurostat.get_data_df('nama_10_gdp')
    print(f"   ✓ Downloaded {len(gdp_country):,} rows")
    
    gdp_country_path = f"{OUTPUT_DIR}/eurostat_gdp_country_raw.csv"
    gdp_country.to_csv(gdp_country_path, index=False)
    print(f"   ✓ Saved: eurostat_gdp_country_raw.csv")
    
except Exception as e:
    print(f"   ✗ Error: {e}")

time.sleep(EUROSTAT_DELAY)

# ============================================================================
# Dataset 4: Crime statistics at NUTS2 level
# ============================================================================
print("\n4. Crime statistics at NUTS2 level (crim_gen_reg)...")
try:
    crime_nuts2 = eurostat.get_data_df('crim_gen_reg')
    print(f"   ✓ Downloaded {len(crime_nuts2):,} rows")
    
    crime_path = f"{OUTPUT_DIR}/eurostat_crime_nuts2_raw.csv"
    crime_nuts2.to_csv(crime_path, index=False)
    print(f"   ✓ Saved: eurostat_crime_nuts2_raw.csv")
    
except Exception as e:
    print(f"   ✗ Error: {e}")

time.sleep(EUROSTAT_DELAY)

# ============================================================================
# Dataset 5: Population at NUTS3 level
# ============================================================================
print("\n5. Population at NUTS3 level (demo_r_pjangrp3)...")
try:
    pop_nuts3 = eurostat.get_data_df('demo_r_pjangrp3')
    print(f"   ✓ Downloaded {len(pop_nuts3):,} rows")
    
    pop_path = f"{OUTPUT_DIR}/eurostat_population_nuts3_raw.csv"
    pop_nuts3.to_csv(pop_path, index=False)
    print(f"   ✓ Saved: eurostat_population_nuts3_raw.csv")
    
except Exception as e:
    print(f"   ✗ Error: {e}")

print(f"\n{'=' * 80}")
print("✓ ALL EUROSTAT DATASETS DOWNLOADED")
print(f"{'=' * 80}")

In [ ]:
# =============================================================================
# CELL 5: Load Main Dataset and Prepare for Enrichment
# =============================================================================

print("\n" + "=" * 80)
print("LOADING MAIN DATASET")
print("=" * 80)

df_main = pd.read_csv(ETER_ENRICHED_PATH)
print(f"✓ Dataset loaded: {len(df_main):,} records")
print(f"  Columns: {len(df_main.columns)}")

# Extract year from Mobility Start Month (format: YYYY-MM)
print("\nExtracting year from 'Mobility Start Month'...")
df_main['Year'] = df_main['Mobility Start Month'].str.split('-').str[0]
df_main['Year'] = pd.to_numeric(df_main['Year'], errors='coerce').astype('Int64')

print(f"✓ Year extracted")
print(f"  Year range: {df_main['Year'].min()} - {df_main['Year'].max()}")
print(f"  Non-null years: {df_main['Year'].notna().sum():,} ({df_main['Year'].notna().sum()/len(df_main)*100:.1f}%)")

# Create NUTS2 and NUTS0 codes from NUTS3
print("\nDeriving NUTS2 and NUTS0 codes from NUTS3...")

def extract_nuts2(nuts3):
    """Extract NUTS2 code from NUTS3 (first 4 characters)"""
    if pd.isna(nuts3):
        return None
    s = str(nuts3)
    return s[:4] if len(s) >= 4 else None

df_main['Sending_NUTS2'] = df_main['Sending_NUTS3'].apply(extract_nuts2)
df_main['Sending_NUTS0'] = df_main['Sending_NUTS3'].str[:2]
df_main['Receiving_NUTS2'] = df_main['Receiving_NUTS3'].apply(extract_nuts2)
df_main['Receiving_NUTS0'] = df_main['Receiving_NUTS3'].str[:2]

print(f"✓ NUTS hierarchy created")
print(f"  Sending NUTS2:  {df_main['Sending_NUTS2'].notna().sum():,}")
print(f"  Sending NUTS0:  {df_main['Sending_NUTS0'].notna().sum():,}")
print(f"  Receiving NUTS2: {df_main['Receiving_NUTS2'].notna().sum():,}")
print(f"  Receiving NUTS0: {df_main['Receiving_NUTS0'].notna().sum():,}")

In [ ]:
# =============================================================================
# CELL 6: Process GDP NUTS3 Data
# =============================================================================

print("\n" + "=" * 80)
print("PROCESSING GDP NUTS3 DATA")
print("=" * 80)

# Load raw data
df_gdp_nuts3 = pd.read_csv(f"{OUTPUT_DIR}/eurostat_gdp_nuts3_raw.csv")

# Fix column name (Eurostat uses geo\TIME_PERIOD)
df_gdp_nuts3 = df_gdp_nuts3.rename(columns={'geo\\TIME_PERIOD': 'geo'})

print(f"Raw data: {len(df_gdp_nuts3):,} rows")

# CRITICAL: Filter to EUR_HAB (euros per inhabitant) BEFORE melting
print("\nFiltering to unit='EUR_HAB' (GDP per capita in euros)...")
df_gdp_nuts3 = df_gdp_nuts3[df_gdp_nuts3['unit'] == 'EUR_HAB'].copy()
print(f"✓ Filtered to {len(df_gdp_nuts3):,} rows")

# Melt wide format to long format (years as rows)
year_cols = [col for col in df_gdp_nuts3.columns if col.isdigit()]
print(f"\nYear columns: {min(year_cols)} to {max(year_cols)}")

df_gdp_nuts3_long = df_gdp_nuts3[['geo'] + year_cols].melt(
    id_vars=['geo'], 
    value_vars=year_cols, 
    var_name='Year', 
    value_name='GDP_NUTS3'
)

df_gdp_nuts3_long['Year'] = pd.to_numeric(df_gdp_nuts3_long['Year']).astype('Int64')
df_gdp_nuts3_long['GDP_NUTS3'] = pd.to_numeric(df_gdp_nuts3_long['GDP_NUTS3'], errors='coerce')

# Remove duplicates
df_gdp_nuts3_long = df_gdp_nuts3_long.drop_duplicates(['geo', 'Year'])
df_gdp_nuts3_long = df_gdp_nuts3_long.rename(columns={'geo': 'NUTS3'})

print(f"✓ Processed to long format: {len(df_gdp_nuts3_long):,} rows")
print(f"  Unique (NUTS3, Year) pairs: {len(df_gdp_nuts3_long):,}")
print(f"  NUTS3 regions: {df_gdp_nuts3_long['NUTS3'].nunique()}")

In [ ]:
# =============================================================================
# CELL 7: Process GDP NUTS2 Data
# =============================================================================

print("\n" + "=" * 80)
print("PROCESSING GDP NUTS2 DATA")
print("=" * 80)

df_gdp_nuts2 = pd.read_csv(f"{OUTPUT_DIR}/eurostat_gdp_nuts2_raw.csv")
df_gdp_nuts2 = df_gdp_nuts2.rename(columns={'geo\\TIME_PERIOD': 'geo'})

print(f"Raw data: {len(df_gdp_nuts2):,} rows")

# Filter to EUR_HAB BEFORE melting
df_gdp_nuts2 = df_gdp_nuts2[df_gdp_nuts2['unit'] == 'EUR_HAB'].copy()
print(f"✓ Filtered to EUR_HAB: {len(df_gdp_nuts2):,} rows")

# Melt to long format
year_cols = [col for col in df_gdp_nuts2.columns if col.isdigit()]
df_gdp_nuts2_long = df_gdp_nuts2[['geo'] + year_cols].melt(
    id_vars=['geo'], 
    value_vars=year_cols, 
    var_name='Year', 
    value_name='GDP_NUTS2'
)

df_gdp_nuts2_long['Year'] = pd.to_numeric(df_gdp_nuts2_long['Year']).astype('Int64')
df_gdp_nuts2_long['GDP_NUTS2'] = pd.to_numeric(df_gdp_nuts2_long['GDP_NUTS2'], errors='coerce')
df_gdp_nuts2_long = df_gdp_nuts2_long.drop_duplicates(['geo', 'Year'])
df_gdp_nuts2_long = df_gdp_nuts2_long.rename(columns={'geo': 'NUTS2'})

print(f"✓ Processed: {len(df_gdp_nuts2_long):,} rows")
print(f"  NUTS2 regions: {df_gdp_nuts2_long['NUTS2'].nunique()}")

In [ ]:
# =============================================================================
# CELL 8: Process GDP Country Data (PPS for Cross-Country Comparison)
# =============================================================================

print("\n" + "=" * 80)
print("PROCESSING GDP COUNTRY DATA")
print("=" * 80)

df_gdp_country = pd.read_csv(f"{OUTPUT_DIR}/eurostat_gdp_country_raw.csv")
df_gdp_country = df_gdp_country.rename(columns={'geo\\TIME_PERIOD': 'geo'})

print(f"Raw data: {len(df_gdp_country):,} rows")

# Use Purchasing Power Standard (PPS) for cross-country comparability
# Note: EUR_HAB not available at country level
print("\nUsing CP_MPPS_EU27_2020 (PPS per capita - comparable across countries)...")
df_gdp_country = df_gdp_country[
    (df_gdp_country['unit'] == 'CP_MPPS_EU27_2020') & 
    (df_gdp_country['na_item'] == 'B1GQ')
].copy()

print(f"✓ Filtered to {len(df_gdp_country):,} rows")

if len(df_gdp_country) == 0:
    print("⚠️  WARNING: No data after filtering!")
    print("   Country-level GDP will not be available")
else:
    # Melt to long format
    year_cols = [col for col in df_gdp_country.columns if col.isdigit()]
    df_gdp_country_long = df_gdp_country[['geo'] + year_cols].melt(
        id_vars=['geo'], 
        value_vars=year_cols, 
        var_name='Year', 
        value_name='GDP_Country_PPS'
    )
    
    df_gdp_country_long['Year'] = pd.to_numeric(df_gdp_country_long['Year']).astype('Int64')
    df_gdp_country_long['GDP_Country_PPS'] = pd.to_numeric(
        df_gdp_country_long['GDP_Country_PPS'], errors='coerce'
    )
    df_gdp_country_long = df_gdp_country_long.drop_duplicates(['geo', 'Year'])
    df_gdp_country_long = df_gdp_country_long.rename(columns={'geo': 'NUTS0'})
    
    print(f"✓ Processed: {len(df_gdp_country_long):,} rows")
    print(f"  Countries: {df_gdp_country_long['NUTS0'].nunique()}")

In [ ]:
# =============================================================================
# CELL 9: Process Crime Data (NUTS2)
# =============================================================================

print("\n" + "=" * 80)
print("PROCESSING CRIME DATA (NUTS2)")
print("=" * 80)

df_crime = pd.read_csv(f"{OUTPUT_DIR}/eurostat_crime_nuts2_raw.csv")
df_crime = df_crime.rename(columns={'geo\\TIME_PERIOD': 'geo'})

print(f"Raw data: {len(df_crime):,} rows")

# Filter to total recorded offences (ICCS0101) per 100k inhabitants
print("\nFiltering to ICCS0101 (total offences) per 100k inhabitants...")
df_crime = df_crime[
    (df_crime['iccs'] == 'ICCS0101') & 
    (df_crime['unit'] == 'P_HTHAB')
].copy()

print(f"✓ Filtered to {len(df_crime):,} rows")

# Melt to long format
year_cols = [col for col in df_crime.columns if col.isdigit()]
df_crime_long = df_crime[['geo'] + year_cols].melt(
    id_vars=['geo'], 
    value_vars=year_cols, 
    var_name='Year', 
    value_name='Crime_rate'
)

df_crime_long['Year'] = pd.to_numeric(df_crime_long['Year']).astype('Int64')
df_crime_long['Crime_rate'] = pd.to_numeric(df_crime_long['Crime_rate'], errors='coerce')
df_crime_long = df_crime_long.drop_duplicates(['geo', 'Year'])
df_crime_long = df_crime_long.rename(columns={'geo': 'NUTS2'})

print(f"✓ Processed: {len(df_crime_long):,} rows")
print(f"  NUTS2 regions: {df_crime_long['NUTS2'].nunique()}")

In [ ]:
# =============================================================================
# CELL 10: Process Population Data (NUTS3)
# =============================================================================

print("\n" + "=" * 80)
print("PROCESSING POPULATION DATA (NUTS3)")
print("=" * 80)

df_pop = pd.read_csv(f"{OUTPUT_DIR}/eurostat_population_nuts3_raw.csv")
df_pop = df_pop.rename(columns={'geo\\TIME_PERIOD': 'geo'})

print(f"Raw data: {len(df_pop):,} rows")

# Filter to total population (all ages, both sexes)
print("\nFiltering to total population (age=TOTAL, sex=T)...")
df_pop = df_pop[
    (df_pop['age'] == 'TOTAL') & 
    (df_pop['sex'] == 'T') & 
    (df_pop['unit'] == 'NR')
].copy()

print(f"✓ Filtered to {len(df_pop):,} rows")

# Melt to long format
year_cols = [col for col in df_pop.columns if col.isdigit()]
df_pop_long = df_pop[['geo'] + year_cols].melt(
    id_vars=['geo'], 
    value_vars=year_cols, 
    var_name='Year', 
    value_name='Population'
)

df_pop_long['Year'] = pd.to_numeric(df_pop_long['Year']).astype('Int64')
df_pop_long['Population'] = pd.to_numeric(df_pop_long['Population'], errors='coerce')
df_pop_long = df_pop_long.drop_duplicates(['geo', 'Year'])
df_pop_long = df_pop_long.rename(columns={'geo': 'NUTS3'})

print(f"✓ Processed: {len(df_pop_long):,} rows")
print(f"  NUTS3 regions: {df_pop_long['NUTS3'].nunique()}")

In [ ]:
# =============================================================================
# CELL 11: Merge Regional Data - SENDING Locations
# =============================================================================

print("\n" + "=" * 80)
print("MERGING REGIONAL DATA - SENDING LOCATIONS")
print("=" * 80)

df_enriched = df_main.copy()
initial_rows = len(df_enriched)
print(f"Starting with {initial_rows:,} rows\n")

# Merge GDP NUTS3
df_enriched = df_enriched.merge(
    df_gdp_nuts3_long.add_prefix('Sending_'),
    left_on=['Sending_NUTS3', 'Year'],
    right_on=['Sending_NUTS3', 'Sending_Year'],
    how='left'
).drop(columns=['Sending_Year'], errors='ignore')
print(f"1. GDP NUTS3:  {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

# Merge GDP NUTS2
df_enriched = df_enriched.merge(
    df_gdp_nuts2_long.add_prefix('Sending_'),
    left_on=['Sending_NUTS2', 'Year'],
    right_on=['Sending_NUTS2', 'Sending_Year'],
    how='left'
).drop(columns=['Sending_Year'], errors='ignore')
print(f"2. GDP NUTS2:  {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

# Merge GDP Country
if 'df_gdp_country_long' in locals():
    df_enriched = df_enriched.merge(
        df_gdp_country_long.add_prefix('Sending_'),
        left_on=['Sending_NUTS0', 'Year'],
        right_on=['Sending_NUTS0', 'Sending_Year'],
        how='left'
    ).drop(columns=['Sending_Year'], errors='ignore')
    print(f"3. GDP Country: {len(df_enriched):,} rows " +
          f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")
else:
    print(f"3. GDP Country: Skipped (no data)")

# Merge Crime
df_enriched = df_enriched.merge(
    df_crime_long.add_prefix('Sending_'),
    left_on=['Sending_NUTS2', 'Year'],
    right_on=['Sending_NUTS2', 'Sending_Year'],
    how='left'
).drop(columns=['Sending_Year'], errors='ignore')
print(f"4. Crime:      {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

# Merge Population
df_enriched = df_enriched.merge(
    df_pop_long.add_prefix('Sending_'),
    left_on=['Sending_NUTS3', 'Year'],
    right_on=['Sending_NUTS3', 'Sending_Year'],
    how='left'
).drop(columns=['Sending_Year'], errors='ignore')
print(f"5. Population: {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

if len(df_enriched) == initial_rows:
    print(f"\n✅ SENDING MERGE SUCCESSFUL - NO ROW EXPLOSION")
else:
    print(f"\n⚠️ WARNING: Row count changed!")

In [ ]:
# =============================================================================
# CELL 12: Merge Regional Data - RECEIVING Locations
# =============================================================================

print("\n" + "=" * 80)
print("MERGING REGIONAL DATA - RECEIVING LOCATIONS")
print("=" * 80)

initial_rows = len(df_enriched)
print(f"Starting with {initial_rows:,} rows\n")

# Merge GDP NUTS3
df_enriched = df_enriched.merge(
    df_gdp_nuts3_long.add_prefix('Receiving_'),
    left_on=['Receiving_NUTS3', 'Year'],
    right_on=['Receiving_NUTS3', 'Receiving_Year'],
    how='left'
).drop(columns=['Receiving_Year'], errors='ignore')
print(f"1. GDP NUTS3:  {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

# Merge GDP NUTS2
df_enriched = df_enriched.merge(
    df_gdp_nuts2_long.add_prefix('Receiving_'),
    left_on=['Receiving_NUTS2', 'Year'],
    right_on=['Receiving_NUTS2', 'Receiving_Year'],
    how='left'
).drop(columns=['Receiving_Year'], errors='ignore')
print(f"2. GDP NUTS2:  {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

# Merge GDP Country
if 'df_gdp_country_long' in locals():
    df_enriched = df_enriched.merge(
        df_gdp_country_long.add_prefix('Receiving_'),
        left_on=['Receiving_NUTS0', 'Year'],
        right_on=['Receiving_NUTS0', 'Receiving_Year'],
        how='left'
    ).drop(columns=['Receiving_Year'], errors='ignore')
    print(f"3. GDP Country: {len(df_enriched):,} rows " +
          f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")
else:
    print(f"3. GDP Country: Skipped (no data)")

# Merge Crime
df_enriched = df_enriched.merge(
    df_crime_long.add_prefix('Receiving_'),
    left_on=['Receiving_NUTS2', 'Year'],
    right_on=['Receiving_NUTS2', 'Receiving_Year'],
    how='left'
).drop(columns=['Receiving_Year'], errors='ignore')
print(f"4. Crime:      {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

# Merge Population
df_enriched = df_enriched.merge(
    df_pop_long.add_prefix('Receiving_'),
    left_on=['Receiving_NUTS3', 'Year'],
    right_on=['Receiving_NUTS3', 'Receiving_Year'],
    how='left'
).drop(columns=['Receiving_Year'], errors='ignore')
print(f"5. Population: {len(df_enriched):,} rows " +
      f"({'✓' if len(df_enriched) == initial_rows else '⚠️ ROW CHANGE'})")

if len(df_enriched) == initial_rows:
    print(f"\n✅ RECEIVING MERGE SUCCESSFUL - NO ROW EXPLOSION")
else:
    print(f"\n⚠️ WARNING: Row count changed!")

In [ ]:
# =============================================================================
# CELL 13: Save Final Dataset and Generate Summary
# =============================================================================

print("\n" + "=" * 80)
print("SAVING FINAL ENRICHED DATASET")
print("=" * 80)

output_file = f"{OUTPUT_DIR}/erasmus_final_complete.csv"
df_enriched.to_csv(output_file, index=False)

print(f"✓ Final dataset saved: erasmus_final_complete.csv")
print(f"  Location: {output_file}")
print(f"  Records: {len(df_enriched):,}")
print(f"  Columns: {len(df_enriched.columns)}")

# Comprehensive coverage summary
print("\n" + "=" * 80)
print("FINAL ENRICHMENT SUMMARY")
print("=" * 80)

print(f"\n📊 DATASET OVERVIEW:")
print(f"  Total records: {len(df_enriched):,}")
print(f"  Total columns: {len(df_enriched.columns)}")
print(f"  Year range: {df_enriched['Year'].min()} - {df_enriched['Year'].max()}")

print(f"\n📍 REGIONAL DATA COVERAGE - SENDING:")
sending_cols = {
    'GDP NUTS3': 'Sending_GDP_NUTS3',
    'GDP NUTS2': 'Sending_GDP_NUTS2',
    'GDP Country (PPS)': 'Sending_GDP_Country_PPS',
    'Crime rate': 'Sending_Crime_rate',
    'Population': 'Sending_Population'
}

for label, col in sending_cols.items():
    if col in df_enriched.columns:
        count = df_enriched[col].notna().sum()
        pct = count / len(df_enriched) * 100
        print(f"  {label:20s}: {count:7,} / {len(df_enriched):,} ({pct:5.1f}%)")

print(f"\n📍 REGIONAL DATA COVERAGE - RECEIVING:")
receiving_cols = {
    'GDP NUTS3': 'Receiving_GDP_NUTS3',
    'GDP NUTS2': 'Receiving_GDP_NUTS2',
    'GDP Country (PPS)': 'Receiving_GDP_Country_PPS',
    'Crime rate': 'Receiving_Crime_rate',
    'Population': 'Receiving_Population'
}

for label, col in receiving_cols.items():
    if col in df_enriched.columns:
        count = df_enriched[col].notna().sum()
        pct = count / len(df_enriched) * 100
        print(f"  {label:20s}: {count:7,} / {len(df_enriched):,} ({pct:5.1f}%)")

# Show sample enriched records
print(f"\n📋 SAMPLE ENRICHED RECORDS:")
print("-" * 80)
sample_cols = [
    'Year', 
    'Sending_NUTS3', 'Sending_GDP_NUTS3', 'Sending_Crime_rate',
    'Receiving_NUTS3', 'Receiving_GDP_NUTS3', 'Receiving_Crime_rate'
]
available_cols = [col for col in sample_cols if col in df_enriched.columns]

sample = df_enriched[df_enriched['Sending_GDP_NUTS3'].notna()].head(5)
print(sample[available_cols].to_string(index=False))

print("\n" + "=" * 80)
print("✅ EUROSTAT REGIONAL ENRICHMENT COMPLETE")
print("=" * 80)
print(f"\nFinal dataset: {len(df_enriched):,} records with {len(df_enriched.columns)} variables")
print(f"Including: Geocoding + NUTS3 + ETER + Eurostat indicators")